# DeBERTa-v3-large ABSA — фолды 1–4


In [ ]:
!pip install -q sentencepiece

In [ ]:
import os, glob
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
np.random.seed(SEED)

DATA = os.path.dirname(glob.glob('/kaggle/input/**/CAIRLAB.csv', recursive=True)[0]) if os.path.exists('/kaggle/input') else '.'
SUB_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else 'submissions'

train_df = pd.read_csv(f'{DATA}/CAIRLAB.csv', header=None, names=['ID', 'Entity', 'Sentiment', 'Text'])
test_df = pd.read_csv(f'{DATA}/test.csv')
train_df['Text'] = train_df['Text'].fillna('')
test_df['Text'] = test_df['Text'].fillna('')
train_df = train_df[train_df['Text'].str.strip() != ''].reset_index(drop=True)

X = train_df['Entity'] + ' ' + train_df['Text']
y = train_df['Sentiment']
LABELS = set(y.unique())

folds = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
            .split(X, y, groups=train_df['ID']))
print('train:', train_df.shape, '| test:', test_df.shape)

In [ ]:
import gc
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

MODEL = 'yangheng/deberta-v3-large-absa-v1.1'
NAME = 'debertaL'
FOLDS = [1, 2, 3, 4]
EPOCHS = 1
BS = 16
LR = 1e-5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
use_amp = device == 'cuda'

classes = sorted(LABELS)
y_num = y.map({c: i for i, c in enumerate(classes)}).values
tok = AutoTokenizer.from_pretrained(MODEL)

def batches(df, shuffle=False):
    idx = np.random.permutation(len(df)) if shuffle else np.arange(len(df))
    for i in range(0, len(df), BS):
        b = idx[i:i + BS]
        enc = tok(list(df['Entity'].iloc[b]), list(df['Text'].iloc[b]),
                  truncation=True, max_length=128, padding=True, return_tensors='pt').to(device)
        yield enc, b

def predict(model, df):
    model.eval()
    out = []
    with torch.no_grad():
        for enc, _ in batches(df):
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
                logits = model(**enc).logits
            out.append(logits.float().softmax(-1).cpu().numpy())
    return np.concatenate(out)

def train_fold(fold):
    tr_idx, va_idx = folds[fold]
    tr_df, va_df = train_df.iloc[tr_idx], train_df.iloc[va_idx]
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=len(classes), ignore_mismatched_sizes=True).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    steps = EPOCHS * (len(tr_df) // BS + 1)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    scaler = torch.amp.GradScaler(enabled=use_amp)

    for epoch in range(EPOCHS):
        model.train()
        for enc, b in tqdm(batches(tr_df, shuffle=True), total=len(tr_df) // BS + 1):
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
                loss = model(**enc, labels=torch.tensor(y_num[tr_idx][b]).to(device)).loss
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            opt.zero_grad()

    val_proba = predict(model, va_df)
    print(f'fold {fold}: val acc = {(val_proba.argmax(1) == y_num[va_idx]).mean():.4f}')
    np.save(f'{SUB_DIR}/{NAME}_f{fold}_val.npy', val_proba)
    np.save(f'{SUB_DIR}/{NAME}_f{fold}_va_idx.npy', va_idx)
    np.save(f'{SUB_DIR}/{NAME}_f{fold}_test.npy', predict(model, test_df))

    del model, opt
    gc.collect()
    torch.cuda.empty_cache()

for fold in FOLDS:
    train_fold(fold)